# YOLO Model Comparison for Dish Detection

This notebook compares three YOLO object detection models to determine the best architecture for detecting dishes in food images. We evaluate:

1. **YOLOv8-n** (nano) - ~3.2M parameters, fastest inference
2. **YOLOv8-s** (small) - ~9.8M parameters, balanced approach
3. **YOLO11-n** (nano) - Latest architecture, improved accuracy

All models are trained on the same dataset with identical hyperparameters for a fair comparison.

## Configuration

Set up training parameters and paths:
- **Image size**: 768px for higher resolution detection
- **Epochs**: 80 training epochs
- **Batch size**: 16 images per batch
- **Device**: GPU (CUDA device 0)

We also reference a pre-trained YOLOv8-n model from earlier training runs for comparison.

In [9]:
from ultralytics import YOLO
import os
import pandas as pd

DATA_YAML = "data_yolo_dish/data.yaml"
IMG_SIZE = 768
EPOCHS = 80
BATCH_SIZE = 16
DEVICE = 0

BEST_YOLOV8N = r"runs/detect/train_dishdet_multi3/weights/best.pt"


## Train YOLOv8-s (Small Variant)

Train YOLOv8-s as a baseline comparison. This model has ~9.8M parameters (3x larger than nano) and typically offers better accuracy at the cost of slower inference speed.

In [10]:
print("=" * 70)
print("Training YOLOv8-s for dish detection")
print("=" * 70)

model_y8s = YOLO("yolov8s.pt")

result_y8s = model_y8s.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    name="compare_yolov8s_dishdet",
    project="runs/detect",
    cache="ram",
    seed=0,
    deterministic=True,
)

BEST_YOLOV8S = os.path.join(
    "runs", "detect", "compare_yolov8s_dishdet", "weights", "best.pt"
)
print(f"\n[YOLOv8-s] Best weights: {BEST_YOLOV8S}")

print("\nValidating best YOLOv8-s weights on validation set")
y8s_best_model = YOLO(BEST_YOLOV8S)
metrics_y8s = y8s_best_model.val(
    data=DATA_YAML,
    split="val",
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
)

print(
    f"[YOLOv8-s] P={metrics_y8s.box.mp:.3f}, "
    f"R={metrics_y8s.box.mr:.3f}, "
    f"mAP50={metrics_y8s.box.map50:.3f}, "
    f"mAP50-95={metrics_y8s.box.map:.3f}"
)


Training YOLOv8-s for dish detection
New https://pypi.org/project/ultralytics/8.3.233 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.5  Python-3.11.9 torch-2.3.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: task=detect, mode=train, model=yolov8s.pt, data=data_yolo_dish/data.yaml, epochs=80, time=None, patience=100, batch=16, imgsz=768, save=True, save_period=-1, cache=ram, device=0, workers=8, project=runs/detect, name=compare_yolov8s_dishdet, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=

AMP: checks passed 


train: Scanning C:\Users\Jun Wei\Desktop\SIT\SIT Y2T1\Computer Vision and Deep Learning\Proj\food_classifier\data_yolo_dish\train\labels.cache... 903 images, 0 backgrounds, 0 corrupt: 100%|██████████| 903/903 [00:00<?, ?it/s]

WARNING  Box and segment counts should be equal, but got len(segments) = 83, len(boxes) = 5212. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.


WARNING  cache='ram' may produce non-deterministic training results. Consider cache='disk' as a deterministic alternative if your disk space allows.


...

                   all        181       1004      0.861      0.898      0.923      0.831
Speed: 0.2ms preprocess, 5.4ms inference, 0.0ms loss, 1.6ms postprocess per image
Results saved to runs\detect\compare_yolov8s_dishdet

[YOLOv8-s] Best weights: runs\detect\compare_yolov8s_dishdet\weights\best.pt

Validating best YOLOv8-s weights on validation set
Ultralytics 8.3.5  Python-3.11.9 torch-2.3.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
Model summary (fused): 186 layers, 9,828,051 parameters, 0 gradients


val: Scanning C:\Users\Jun Wei\Desktop\SIT\SIT Y2T1\Computer Vision and Deep Learning\Proj\food_classifier\data_yolo_dish\valid\labels.cache... 181 images, 0 backgrounds, 0 corrupt: 100%|██████████| 181/181 [00:00<?, ?it/s]

WARNING  Box and segment counts should be equal, but got len(segments) = 28, len(boxes) = 1004. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 12/12 [00:05<00:00,  2.36it/s]


                   all        181       1004      0.864      0.896      0.923      0.831
Speed: 0.5ms preprocess, 8.7ms inference, 0.0ms loss, 2.4ms postprocess per image
Results saved to c:\Users\Jun Wei\Desktop\SIT\SIT Y2T1\Computer Vision and Deep Learning\Proj\food_classifier\runs\detect\val7
[YOLOv8-s] P=0.864, R=0.896, mAP50=0.923, mAP50-95=0.831


## Train YOLO11-n (Latest Architecture)

Train YOLO11-n, the newest YOLO architecture released in late 2024. This model features improved accuracy through architectural enhancements while maintaining a small parameter count.

In [11]:
print("=" * 70)
print("Training YOLO11-n for dish detection")
print("=" * 70)

model_y11n = YOLO("yolo11n.pt")

result_y11n = model_y11n.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    name="compare_yolo11n_dishdet",
    project="runs/detect",
    cache="ram",
    seed=0,
    deterministic=True,
)

BEST_YOLO11N = os.path.join(
    "runs", "detect", "compare_yolo11n_dishdet", "weights", "best.pt"
)
print(f"\n[YOLO11-n] Best weights: {BEST_YOLO11N}")

print("\nValidating best YOLO11-n weights on validation set")
y11n_best_model = YOLO(BEST_YOLO11N)
metrics_y11n = y11n_best_model.val(
    data=DATA_YAML,
    split="val",
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
)

print(
    f"[YOLO11-n] P={metrics_y11n.box.mp:.3f}, "
    f"R={metrics_y11n.box.mr:.3f}, "
    f"mAP50={metrics_y11n.box.map50:.3f}, "
    f"mAP50-95={metrics_y11n.box.map:.3f}"
)


Training YOLO11-n for dish detection
New https://pypi.org/project/ultralytics/8.3.233 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.5  Python-3.11.9 torch-2.3.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: task=detect, mode=train, model=yolo11n.pt, data=data_yolo_dish/data.yaml, epochs=80, time=None, patience=100, batch=16, imgsz=768, save=True, save_period=-1, cache=ram, device=0, workers=8, project=runs/detect, name=compare_yolo11n_dishdet, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=

train: Scanning C:\Users\Jun Wei\Desktop\SIT\SIT Y2T1\Computer Vision and Deep Learning\Proj\food_classifier\data_yolo_dish\train\labels.cache... 903 images, 0 backgrounds, 0 corrupt: 100%|██████████| 903/903 [00:00<?, ?it/s]

WARNING  Box and segment counts should be equal, but got len(segments) = 83, len(boxes) = 5212. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
train: 1.5GB RAM required to cache images with 50% safety margin but only 1.2/31.7GB available, not caching images 



val: Scanning C:\Users\Jun Wei\Desktop\SIT\SIT Y2T1\Computer Vision and Deep Learning\Proj\food_classifier\data_yolo_dish\valid\labels.cache... 181 images, 0 backgrounds, 0 corrupt: 100%|██████████| 181/181 [00:00<?, ?it/s]

WARNING  Box and segment counts should be equal, but got len(segments) = 28, len(boxes) = 1004. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
WARNING  cache='ram' may produce non-deterministic training results. Consider cache='disk' as a deterministic alternative if your disk space allows.


...

                   all        181       1004      0.884      0.892      0.929      0.847
Speed: 0.2ms preprocess, 2.2ms inference, 0.0ms loss, 1.9ms postprocess per image
Results saved to runs\detect\compare_yolo11n_dishdet

[YOLO11-n] Best weights: runs\detect\compare_yolo11n_dishdet\weights\best.pt

Validating best YOLO11-n weights on validation set
Ultralytics 8.3.5  Python-3.11.9 torch-2.3.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
YOLO11n summary (fused): 238 layers, 2,582,347 parameters, 0 gradients


val: Scanning C:\Users\Jun Wei\Desktop\SIT\SIT Y2T1\Computer Vision and Deep Learning\Proj\food_classifier\data_yolo_dish\valid\labels.cache... 181 images, 0 backgrounds, 0 corrupt: 100%|██████████| 181/181 [00:00<?, ?it/s]

WARNING  Box and segment counts should be equal, but got len(segments) = 28, len(boxes) = 1004. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 12/12 [00:03<00:00,  3.22it/s]


                   all        181       1004      0.884      0.892      0.928      0.847
Speed: 0.6ms preprocess, 4.8ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to c:\Users\Jun Wei\Desktop\SIT\SIT Y2T1\Computer Vision and Deep Learning\Proj\food_classifier\runs\detect\val8
[YOLO11-n] P=0.884, R=0.892, mAP50=0.928, mAP50-95=0.847


## Model Comparison

Evaluate all three models on the validation set and compare their performance metrics:
- **Precision (P)**: How many detected dishes are correct
- **Recall (R)**: How many actual dishes are detected
- **mAP50**: Mean Average Precision at IoU threshold 0.5
- **mAP50-95**: Mean Average Precision averaged across IoU thresholds 0.5-0.95

In [12]:
print("=" * 70)
print("Comparing YOLOv8-n vs YOLOv8-s vs YOLO11-n")
print("=" * 70)

# YOLOv8-n
model_y8n = YOLO(BEST_YOLOV8N)
metrics_y8n = model_y8n.val(
    data=DATA_YAML,
    split="val",
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    verbose=False,
)

# YOLOv8-s
metrics_y8s = y8s_best_model.val(
    data=DATA_YAML,
    split="val",
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    verbose=False,
)

# YOLO11-n
metrics_y11n = y11n_best_model.val(
    data=DATA_YAML,
    split="val",
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    verbose=False,
)

rows = [
    {
        "Model": "YOLOv8-n",
        "Params": "~3.2M (tiny)",
        "P": metrics_y8n.box.mp,
        "R": metrics_y8n.box.mr,
        "mAP50": metrics_y8n.box.map50,
        "mAP50-95": metrics_y8n.box.map,
    },
    {
        "Model": "YOLOv8-s",
        "Params": "~9.8M (small)",
        "P": metrics_y8s.box.mp,
        "R": metrics_y8s.box.mr,
        "mAP50": metrics_y8s.box.map50,
        "mAP50-95": metrics_y8s.box.map,
    },
    {
        "Model": "YOLO11-n",
        "Params": "~[n-size] (nano)",
        "P": metrics_y11n.box.mp,
        "R": metrics_y11n.box.mr,
        "mAP50": metrics_y11n.box.map50,
        "mAP50-95": metrics_y11n.box.map,
    },
]

df_compare = pd.DataFrame(rows)
df_compare


Comparing YOLOv8-n vs YOLOv8-s vs YOLO11-n
Ultralytics 8.3.5  Python-3.11.9 torch-2.3.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
Model summary (fused): 186 layers, 2,684,563 parameters, 0 gradients


val: Scanning C:\Users\Jun Wei\Desktop\SIT\SIT Y2T1\Computer Vision and Deep Learning\Proj\food_classifier\data_yolo_dish\valid\labels.cache... 181 images, 0 backgrounds, 0 corrupt: 100%|██████████| 181/181 [00:00<?, ?it/s]

WARNING  Box and segment counts should be equal, but got len(segments) = 28, len(boxes) = 1004. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 12/12 [00:04<00:00,  2.94it/s]


                   all        181       1004      0.852      0.905      0.917      0.828
Speed: 0.7ms preprocess, 4.5ms inference, 0.0ms loss, 1.9ms postprocess per image
Results saved to c:\Users\Jun Wei\Desktop\SIT\SIT Y2T1\Computer Vision and Deep Learning\Proj\food_classifier\runs\detect\val9
Ultralytics 8.3.5  Python-3.11.9 torch-2.3.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)


...

val: Scanning C:\Users\Jun Wei\Desktop\SIT\SIT Y2T1\Computer Vision and Deep Learning\Proj\food_classifier\data_yolo_dish\valid\labels.cache... 181 images, 0 backgrounds, 0 corrupt: 100%|██████████| 181/181 [00:00<?, ?it/s]

WARNING  Box and segment counts should be equal, but got len(segments) = 28, len(boxes) = 1004. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 12/12 [00:04<00:00,  2.83it/s]


                   all        181       1004      0.884      0.892      0.928      0.847
Speed: 0.5ms preprocess, 5.5ms inference, 0.0ms loss, 1.7ms postprocess per image
Results saved to c:\Users\Jun Wei\Desktop\SIT\SIT Y2T1\Computer Vision and Deep Learning\Proj\food_classifier\runs\detect\val11


,Model,Params,P,R,mAP50,mAP50-95
0,YOLOv8-n,~3.2M (tiny),0.852026,0.905378,0.917199,0.827593
1,YOLOv8-s,~9.8M (small),0.864029,0.896414,0.922875,0.830966
2,YOLO11-n,~[n-size] (nano),0.883558,0.891817,0.928259,0.846787


## Results & Conclusion

### Performance Summary

| Model | Params | P | R | mAP50 | mAP50-95 | Inference |
|-------|--------|---|---|-------|----------|-----------|
| YOLOv8-n | ~3.2M | 0.852 | 0.905 | 0.917 | 0.828 | 4.5ms |
| YOLOv8-s | ~9.8M | 0.864 | 0.896 | 0.923 | 0.831 | 8.8ms |
| YOLO11-n | ~2.6M | 0.884 | 0.892 | 0.928 | 0.847 | 5.5ms |

### Why We Choose YOLOv8-n

Although YOLO11-n achieves slightly higher accuracy metrics, we select **YOLOv8-n** for deployment based on:

1. **Fastest inference (4.5ms)** - Critical for real-time webcam streaming in our Gradio app
2. **Highest recall (0.905)** - Detects more dishes with fewer misses, important for food classification pipeline
3. **Production stability** - YOLOv8 is more mature with better documentation and community support
4. **Marginal accuracy difference** - Only ~1-2% behind YOLO11-n, negligible for our use case

For applications requiring maximum accuracy over speed, YOLO11-n would be the better choice.